# Final Stage F: Revised Direct 18-Mask Comparator

**Runtime:** CPU. Select **Runtime > Run all**.

This notebook does not index the full 51R2 or 53R2 capture directories. It joins the existing lightweight 51R2 and 53R2 call traces by `page + call_number`, identifies the nine positions whose semantic Boolean-mask hashes differ, constructs the corresponding eighteen `mask.npy` paths directly, stages only those files to `/content`, compares coordinates pair by pair, and maps every differing coordinate to persisted Notebook45 regional `M3.png` ownership.

It fails closed if the traces do not produce exactly nine differing call positions or if any directly constructed mask path is missing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q opencv-python-headless pandas numpy

## 1. Configuration

Normally no path edits are required. If your outputs live elsewhere, set the three overrides to exact paths.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import hashlib, json, platform, re, shutil, sys

import cv2
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# Optional overrides
TRACE_51_CSV_OVERRIDE = None
TRACE_53_CSV_OVERRIDE = None
CAPTURE_51_ROOT_OVERRIDE = None
CAPTURE_53_ROOT_OVERRIDE = None
NOTEBOOK45_ROOT_OVERRIDE = None

TRACE_BASE = ROOT / 'stage_f_seven_page_merge_trace'
CAPTURE_BASE = ROOT / 'stage_f_final_call_inputs'
NOTEBOOK45_PARENT = ROOT / 'observed_print_classifier_v8_6_notebook45'

FAILING_PAGES = [
    'Page003', 'Page004', 'Page005', 'Page008',
    'Page036', 'Page040', 'Page049',
]
EXPECTED_DIFFERING_CALLS = 9

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_revised_direct_18_mask_comparator'
OUT = ROOT / 'stage_f_revised_direct_18_mask_comparator' / RUN_ID
TABLES = OUT / 'tables'
AUDIT = OUT / 'audit'
IMAGES = OUT / 'images'
LOCAL = Path('/content/stage_f_revised_direct_18_masks')
for d in [OUT, TABLES, AUDIT, IMAGES]:
    d.mkdir(parents=True, exist_ok=False)
if LOCAL.exists():
    shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True, exist_ok=False)

print('Output:', OUT)

## 2. Helpers

In [ ]:
def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def page_of(value):
    match = re.search(r'Page\d+', str(value), flags=re.IGNORECASE)
    if not match:
        return None
    return 'Page' + match.group(0)[4:].zfill(3)


def semantic_hash(array):
    array = np.ascontiguousarray(array)
    h = hashlib.sha256()
    h.update(str(array.dtype).encode())
    h.update(str(array.shape).encode())
    h.update(array.tobytes())
    return h.hexdigest()


def boolean_hash(array):
    return semantic_hash((np.asarray(array) > 0).astype(np.uint8))


def latest_direct_child(root, required_relative_paths):
    root = Path(root)
    if not root.exists():
        return None
    candidates = []
    for child in root.iterdir():
        if child.is_dir() and all((child / r).exists() for r in required_relative_paths):
            candidates.append(child)
    return max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None


def find_column(frame, names):
    lower = {str(c).lower(): c for c in frame.columns}
    for name in names:
        if name.lower() in lower:
            return lower[name.lower()]
    return None


def canonical_trace(frame, side):
    page_col = find_column(frame, ['page'])
    call_col = find_column(frame, ['call_number'])
    hash_col = find_column(frame, ['mask_boolean_sha256'])
    pixel_col = find_column(frame, ['mask_pixels'])
    flag_col = find_column(frame, ['flags'])
    flag_name_col = find_column(frame, ['flags_name'])
    require(page_col and call_col and hash_col,
            f'{side}: trace must contain page, call_number and mask_boolean_sha256. Columns: {frame.columns.tolist()}')
    out = pd.DataFrame({
        'page': frame[page_col].map(page_of),
        'call_number': pd.to_numeric(frame[call_col], errors='raise').astype(int),
        f'mask_hash_{side}': frame[hash_col].astype(str).str.strip(),
        f'mask_pixels_{side}': pd.to_numeric(frame[pixel_col], errors='coerce') if pixel_col else np.nan,
    })
    if flag_col:
        flags = pd.to_numeric(frame[flag_col], errors='coerce')
        out = out.loc[flags == int(cv2.INPAINT_TELEA)].copy()
    elif flag_name_col:
        out = out.loc[frame[flag_name_col].astype(str).str.upper().str.contains('TELEA').to_numpy()].copy()
    out = out[out.page.isin(FAILING_PAGES)].copy()
    require(not out.empty, f'{side}: no target-page Telea rows remain.')
    require(not out.duplicated(['page', 'call_number']).any(),
            f'{side}: duplicate page + call_number rows detected.')
    return out.sort_values(['page', 'call_number']).reset_index(drop=True)

## 3. Resolve the two lightweight call traces

This checks only the known trace locations. It does not recurse through capture call directories.

In [ ]:
TRACE_51 = Path(TRACE_51_CSV_OVERRIDE) if TRACE_51_CSV_OVERRIDE else TRACE_BASE / '51R2' / 'inpaint_call_trace.csv'
TRACE_53 = Path(TRACE_53_CSV_OVERRIDE) if TRACE_53_CSV_OVERRIDE else TRACE_BASE / '53R2' / 'inpaint_call_trace.csv'

require(TRACE_51.exists(), f'Missing 51R2 trace: {TRACE_51}')
require(TRACE_53.exists(), f'Missing 53R2 trace: {TRACE_53}')

raw_51 = pd.read_csv(TRACE_51, dtype=str, keep_default_na=False)
raw_53 = pd.read_csv(TRACE_53, dtype=str, keep_default_na=False)
trace_51 = canonical_trace(raw_51, '51R2')
trace_53 = canonical_trace(raw_53, '53R2')

print('51R2 target calls:', len(trace_51))
print('53R2 target calls:', len(trace_53))
print('51R2 by page:')
display(trace_51.groupby('page').size().rename('calls').reset_index())
print('53R2 by page:')
display(trace_53.groupby('page').size().rename('calls').reset_index())

## 4. Join by page and call number and select the nine differing hashes

In [ ]:
joined = trace_51.merge(
    trace_53,
    on=['page', 'call_number'],
    how='outer',
    indicator=True,
    validate='one_to_one',
)

join_failures = joined[joined['_merge'] != 'both'].copy()
join_failures.to_csv(AUDIT / 'unmatched_page_call_number_rows.csv', index=False)
require(join_failures.empty,
        f'Trace rosters do not align by page + call_number. See {AUDIT / "unmatched_page_call_number_rows.csv"}')

joined['mask_hash_exact'] = joined['mask_hash_51R2'] == joined['mask_hash_53R2']
differing = joined.loc[~joined['mask_hash_exact']].copy()
joined.to_csv(TABLES / 'all_joined_call_hashes.csv', index=False)
differing.to_csv(TABLES / 'nine_differing_call_positions.csv', index=False)

print('Joined calls:', len(joined))
print('Exact hashes:', int(joined.mask_hash_exact.sum()))
print('Differing hashes:', len(differing))
display(differing[['page','call_number','mask_pixels_51R2','mask_pixels_53R2','mask_hash_51R2','mask_hash_53R2']])

require(len(differing) == EXPECTED_DIFFERING_CALLS,
        f'Expected exactly {EXPECTED_DIFFERING_CALLS} differing page/call positions; found {len(differing)}. '
        'Do not continue until the trace lineage is resolved.')

expected_by_page = {'Page003':2,'Page004':2,'Page005':1,'Page008':1,'Page036':1,'Page040':1,'Page049':1}
actual_by_page = differing.groupby('page').size().to_dict()
require(actual_by_page == expected_by_page,
        f'Unexpected differing-call distribution. Expected {expected_by_page}; found {actual_by_page}.')

## 5. Construct the eighteen mask paths directly

The call directory is deterministic: `calls/call_XXXXXX/mask.npy`. No directory index or `rglob` is used.

In [ ]:
CAPTURE_51 = Path(CAPTURE_51_ROOT_OVERRIDE) if CAPTURE_51_ROOT_OVERRIDE else CAPTURE_BASE / '51R2'
CAPTURE_53 = Path(CAPTURE_53_ROOT_OVERRIDE) if CAPTURE_53_ROOT_OVERRIDE else CAPTURE_BASE / '53R2'

require(CAPTURE_51.exists(), f'Missing 51R2 capture root: {CAPTURE_51}')
require(CAPTURE_53.exists(), f'Missing 53R2 capture root: {CAPTURE_53}')

path_rows = []
for _, row in differing.iterrows():
    call_name = f"call_{int(row.call_number):06d}"
    p51 = CAPTURE_51 / 'calls' / call_name / 'mask.npy'
    p53 = CAPTURE_53 / 'calls' / call_name / 'mask.npy'
    path_rows.extend([
        {'side':'51R2','page':row.page,'call_number':int(row.call_number),'mask_path':str(p51)},
        {'side':'53R2','page':row.page,'call_number':int(row.call_number),'mask_path':str(p53)},
    ])

paths = pd.DataFrame(path_rows)
paths['exists'] = paths.mask_path.map(lambda x: Path(x).exists())
paths['bytes'] = paths.mask_path.map(lambda x: Path(x).stat().st_size if Path(x).exists() else np.nan)
paths.to_csv(TABLES / 'resolved_18_mask_paths.csv', index=False)

display(paths)
require(len(paths) == 18, f'Expected 18 direct mask paths; found {len(paths)}.')
require(paths.exists.all(),
        'One or more directly constructed mask paths do not exist. Set CAPTURE_51_ROOT_OVERRIDE and '
        'CAPTURE_53_ROOT_OVERRIDE to the capture roots that produced the lightweight traces. '
        f'See {TABLES / "resolved_18_mask_paths.csv"}')

## 6. Stage only the eighteen masks to local Colab storage and verify hashes

In [ ]:
staged_rows = []
for _, row in paths.sort_values(['page','call_number','side']).iterrows():
    source = Path(row.mask_path)
    destination = LOCAL / f"{row.side}_{row.page}_call_{int(row.call_number):06d}_mask.npy"
    shutil.copy2(source, destination)
    mask = np.load(destination, mmap_mode='r', allow_pickle=False)
    actual_hash = boolean_hash(mask)
    expected_hash = differing.loc[
        (differing.page == row.page) & (differing.call_number == int(row.call_number)),
        'mask_hash_51R2' if row.side == '51R2' else 'mask_hash_53R2'
    ].iloc[0]
    staged_rows.append({
        **row.to_dict(),
        'local_mask_path': str(destination),
        'actual_boolean_sha256': actual_hash,
        'expected_boolean_sha256': expected_hash,
        'hash_exact': actual_hash == expected_hash,
        'shape': str(tuple(mask.shape)),
        'mask_pixels_actual': int(np.count_nonzero(mask)),
    })
    del mask

staged = pd.DataFrame(staged_rows)
staged.to_csv(TABLES / 'staged_18_masks.csv', index=False)
require(staged.hash_exact.all(),
        f'At least one staged mask does not match its trace hash. See {TABLES / "staged_18_masks.csv"}')
print('Staged exactly', len(staged), 'mask files locally.')
print('Total Drive bytes read:', int(staged.bytes.sum()))

## 7. Compare the nine aligned mask pairs

In [ ]:
pair_rows = []
raw_pairs = []
for _, row in differing.sort_values(['page','call_number']).iterrows():
    page = row.page
    call = int(row.call_number)
    lpath = Path(staged.loc[(staged.side=='51R2') & (staged.page==page) & (staged.call_number==call), 'local_mask_path'].iloc[0])
    rpath = Path(staged.loc[(staged.side=='53R2') & (staged.page==page) & (staged.call_number==call), 'local_mask_path'].iloc[0])
    m51 = np.asarray(np.load(lpath, mmap_mode='r', allow_pickle=False) > 0)
    m53 = np.asarray(np.load(rpath, mmap_mode='r', allow_pickle=False) > 0)
    require(m51.shape == m53.shape, f'{page} call {call}: mask shape mismatch {m51.shape} vs {m53.shape}')
    shared = m51 & m53
    only51 = m51 & ~m53
    only53 = m53 & ~m51
    xor = m51 ^ m53
    pair_id = f'{page}_call_{call:06d}'
    d = IMAGES / pair_id
    d.mkdir(parents=True, exist_ok=False)
    for name, mask in [('mask_51',m51),('mask_53',m53),('shared',shared),('only_51',only51),('only_53',only53),('xor',xor)]:
        require(cv2.imwrite(str(d / f'{name}.png'), mask.astype(np.uint8)*255), f'Could not write {d/name}')
    pair_rows.append({
        'pair_id': pair_id,
        'page': page,
        'call_number': call,
        'mask_pixels_51': int(m51.sum()),
        'mask_pixels_53': int(m53.sum()),
        'shared_pixels': int(shared.sum()),
        'only_51_pixels': int(only51.sum()),
        'only_53_pixels': int(only53.sum()),
        'xor_pixels': int(xor.sum()),
        'mask_51_sha256': semantic_hash(m51.astype(np.uint8)),
        'mask_53_sha256': semantic_hash(m53.astype(np.uint8)),
    })
    raw_pairs.append({'pair_id':pair_id,'page':page,'call_number':call,'m51':m51,'m53':m53,'only51':only51,'only53':only53})

pairs = pd.DataFrame(pair_rows)
pairs.to_csv(TABLES / 'exceptional_mask_pairs.csv', index=False)
display(pairs)

## 8. Resolve Notebook45 by shallow run inspection

This looks only at direct children of the expected Notebook45 parent and the seven required page directories.

In [ ]:
def complete_n45_run(run):
    masks = run / 'masks'
    return run.is_dir() and masks.exists() and all((masks / page).exists() for page in FAILING_PAGES)

if NOTEBOOK45_ROOT_OVERRIDE is not None:
    N45 = Path(NOTEBOOK45_ROOT_OVERRIDE)
    require(complete_n45_run(N45), f'Invalid Notebook45 override: {N45}')
else:
    require(NOTEBOOK45_PARENT.exists(),
            f'Notebook45 parent does not exist: {NOTEBOOK45_PARENT}. Set NOTEBOOK45_ROOT_OVERRIDE.')
    runs = [p for p in NOTEBOOK45_PARENT.iterdir() if complete_n45_run(p)]
    require(runs, f'No complete Notebook45 run found directly below {NOTEBOOK45_PARENT}.')
    N45 = max(runs, key=lambda p: p.stat().st_mtime)

print('Notebook45:', N45)

## 9. Load only the seven pages of persisted Notebook45 regional M3 masks

In [ ]:
regions = defaultdict(list)
inventory = []
for page in FAILING_PAGES:
    page_dir = N45 / 'masks' / page
    region_dirs = [p for p in page_dir.iterdir() if p.is_dir()]
    for region_dir in sorted(region_dirs):
        path = region_dir / 'M3.png'
        if not path.exists():
            continue
        image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        require(image is not None, f'Unreadable Notebook45 mask: {path}')
        mask = image > 0
        regions[page].append({'region_id':region_dir.name,'path':path,'mask':mask})
        inventory.append({
            'page':page,'region_id':region_dir.name,'path':str(path),
            'pixels':int(mask.sum()),'semantic_sha256':semantic_hash(mask.astype(np.uint8)),
        })
    require(regions[page], f'{page}: no M3.png files found under {page_dir}')

pd.DataFrame(inventory).to_csv(AUDIT / 'notebook45_regional_m3_inventory.csv', index=False)
print('Notebook45 regional masks loaded:', len(inventory))

## 10. Map every differing coordinate to Notebook45 ownership

In [ ]:
coordinate_rows = []
ownership_rows = []
for pair in raw_pairs:
    page = pair['page']
    shape = pair['m51'].shape
    for region in regions[page]:
        require(region['mask'].shape == shape,
                f'{page}/{region["region_id"]}: M3 shape {region["mask"].shape} does not match call-mask shape {shape}.')
    for direction, diff in [('ONLY_51', pair['only51']), ('ONLY_53', pair['only53'])]:
        owner_counter = Counter()
        ys, xs = np.where(diff)
        for y, x in zip(ys, xs):
            owners = sorted(region['region_id'] for region in regions[page] if region['mask'][y,x])
            owner_text = '|'.join(owners)
            owner_counter[owner_text] += 1
            coordinate_rows.append({
                'pair_id':pair['pair_id'],'page':page,'call_number':pair['call_number'],
                'direction':direction,'y':int(y),'x':int(x),
                'owner_count':len(owners),'owner_ids':owner_text,
                'single_owner':len(owners)==1,
            })
        for owner_text, count in owner_counter.items():
            ownership_rows.append({
                'pair_id':pair['pair_id'],'page':page,'call_number':pair['call_number'],
                'direction':direction,'owner_ids':owner_text,
                'owner_count':0 if not owner_text else len(owner_text.split('|')),
                'pixels':int(count),
            })

coordinates = pd.DataFrame(coordinate_rows)
ownership = pd.DataFrame(ownership_rows)
require(not coordinates.empty, 'The nine differing masks produced no coordinate differences.')
coordinates.to_csv(TABLES / 'exceptional_mask_coordinate_differences.csv', index=False)
ownership.to_csv(TABLES / 'exceptional_coordinate_ownership_summary.csv', index=False)
display(ownership.sort_values(['page','call_number','direction','pixels'], ascending=[True,True,True,False]))

## 11. Verdict and first-correction guidance

In [ ]:
all_single_owner = bool(coordinates.single_owner.all())
zero_owner = int((coordinates.owner_count == 0).sum())
multi_owner = int((coordinates.owner_count > 1).sum())
only51_total = int((coordinates.direction == 'ONLY_51').sum())
only53_total = int((coordinates.direction == 'ONLY_53').sum())
all_53_supersets = bool((pairs.only_51_pixels == 0).all() and (pairs.only_53_pixels > 0).all())
all_51_supersets = bool((pairs.only_53_pixels == 0).all() and (pairs.only_51_pixels > 0).all())

if all_53_supersets and all_single_owner:
    diagnosis = '53R2_NINE_MASKS_ARE_STRICT_SINGLE_OWNER_SUPERSETS_OF_51R2'
    correction = 'Make 53R2 reproduce the 51R2 regional pixel-assignment rule and remove the reported ONLY_53 coordinates from those nine calls.'
elif all_51_supersets and all_single_owner:
    diagnosis = '51R2_NINE_MASKS_ARE_STRICT_SINGLE_OWNER_SUPERSETS_OF_53R2'
    correction = 'Make 53R2 include the reported ONLY_51 coordinates in those nine calls.'
elif all_single_owner:
    diagnosis = 'BIDIRECTIONAL_SINGLE_OWNER_REGIONAL_ASSIGNMENT_DIFFERENCE'
    correction = 'Reproduce the 51R2 regional assignment rule exactly; the nine calls contain both omissions and additions.'
elif zero_owner:
    diagnosis = 'SOME_DIFFERING_COORDINATES_HAVE_NO_NOTEBOOK45_M3_OWNER'
    correction = 'Resolve the selected Notebook45 run or M3 lineage before editing 53R2.'
else:
    diagnosis = 'MULTI_OWNER_REGIONAL_ASSIGNMENT_AMBIGUITY_REMAINS'
    correction = 'Resolve overlap ownership/order before editing 53R2.'

verdict = {
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'design':'page-call-number trace join; direct eighteen-mask loading',
    'trace_51':str(TRACE_51),'trace_53':str(TRACE_53),
    'capture_51':str(CAPTURE_51),'capture_53':str(CAPTURE_53),
    'notebook45':str(N45),
    'joined_target_calls':int(len(joined)),
    'exact_call_hashes':int(joined.mask_hash_exact.sum()),
    'differing_call_positions':int(len(differing)),
    'mask_files_loaded':18,
    'full_capture_directories_indexed':False,
    'mask_regeneration':False,
    'telea_rerun':False,
    'only_51_coordinates':only51_total,
    'only_53_coordinates':only53_total,
    'all_differing_coordinates_single_owner':all_single_owner,
    'zero_owner_coordinates':zero_owner,
    'multi_owner_coordinates':multi_owner,
    'diagnosis':diagnosis,
    'first_correction':correction,
    'stage_f_may_be_declared_pass':False,
    'stage_f_pass_condition':'Correct 53R2, rerun 53R2 and Notebook54, and require zero Stage F differing pixels.',
    'software':{
        'python':sys.version,'opencv':cv2.__version__,
        'numpy':np.__version__,'pandas':pd.__version__,
        'platform':platform.platform(),
    },
}
(OUT / 'revised_direct_18_mask_verdict.json').write_text(json.dumps(verdict, indent=2))
print(json.dumps(verdict, indent=2))

## 12. Final output inventory

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1048576), b''):
            h.update(block)
    return h.hexdigest()

required = [
    OUT / 'revised_direct_18_mask_verdict.json',
    TABLES / 'nine_differing_call_positions.csv',
    TABLES / 'resolved_18_mask_paths.csv',
    TABLES / 'staged_18_masks.csv',
    TABLES / 'exceptional_mask_pairs.csv',
    TABLES / 'exceptional_mask_coordinate_differences.csv',
    TABLES / 'exceptional_coordinate_ownership_summary.csv',
    AUDIT / 'notebook45_regional_m3_inventory.csv',
]
for path in required:
    require(path.exists(), f'Missing required output: {path}')

inventory = []
for path in sorted(OUT.rglob('*')):
    if path.is_file():
        inventory.append({
            'relative_path':str(path.relative_to(OUT)),
            'bytes':path.stat().st_size,
            'sha256':sha256_file(path),
        })
pd.DataFrame(inventory).to_csv(OUT / 'output_inventory.csv', index=False)

readme = f"""REVISED DIRECT 18-MASK COMPARATOR

Run: {OUT}

Start with:
- revised_direct_18_mask_verdict.json
- tables/nine_differing_call_positions.csv
- tables/exceptional_mask_pairs.csv
- tables/exceptional_mask_coordinate_differences.csv
- tables/exceptional_coordinate_ownership_summary.csv

This run joined lightweight call traces by page + call_number, loaded exactly eighteen mask.npy files, did not index the complete capture directories, did not regenerate masks, and did not rerun Telea.
"""
(OUT / 'README.txt').write_text(readme)
print('COMPLETE:', OUT)